## make point mutation

In [ ]:
from experiments.utils import modify_residues_in_cdr

src_pkl = '/home/psh/benchmark_after210930/meta/8hs2_B_C_R.pkl'
target_dir = '/home/psh/benchmark_after210930/meta_modif'
cdr_sequence = "GAGGFLRIITKFDY"

mutation_list = [[(7, "GLY")], [(8, "GLY")], [(7, "GLY"), (8, "GLY")]]

for mutations in mutation_list:
    modify_residues_in_cdr(src_pkl, target_dir, cdr_sequence, mutations=mutations)

변경: global_idx=105, new_resname=GLY
✅ 저장 완료: /home/psh/benchmark_after210930/meta_modif/8hs2_B_C_R_7GLY.pkl
변경: global_idx=106, new_resname=GLY
✅ 저장 완료: /home/psh/benchmark_after210930/meta_modif/8hs2_B_C_R_8GLY.pkl
변경: global_idx=105, new_resname=GLY
변경: global_idx=106, new_resname=GLY
✅ 저장 완료: /home/psh/benchmark_after210930/meta_modif/8hs2_B_C_R_7GLY_8GLY.pkl


### save point mutation csv file

In [ ]:
import os
import pandas as pd

# 입력
input_csv_file = '/home/psh/benchmark_after210930/meta/metadata.csv'
target_dir = '/home/psh/benchmark_after210930/meta_modif'
target_wt_pdbs = ['8hs2_B_C_R']

# 1. CSV 파일 읽기
df = pd.read_csv(input_csv_file)

# 2. target_wt_pdbs에 해당하는 행 선택
filtered_df = df[df['pdb_name'].isin(target_wt_pdbs)].copy()

# 3. target_dir 내부 파일 리스트 확인
files = os.listdir(target_dir)

# 4. target_wt_pdbs 각각에 대해 포함된 파일 개수를 세고 행 복제
duplicated_rows = []
for pdb_name in target_wt_pdbs:
    # pdb_name이 포함된 파일만 필터링
    matched_files = [f for f in files if pdb_name in f]
    print(f"{pdb_name} 관련 파일 개수: {len(matched_files)}")

    if len(matched_files) > 0:
        # 해당 pdb_name 행 가져오기
        rows = filtered_df[filtered_df['pdb_name'] == pdb_name]
        base_row = rows.iloc[0]  # 보통 하나일 것이므로 첫 행 기준
        
        # 각 파일마다 하나의 행 생성
        for fname in matched_files:
            new_row = base_row.copy()
            new_row['processed_path'] = os.path.join(target_dir, fname)
            duplicated_rows.append(new_row)

# 5. 결과 DataFrame 생성
if duplicated_rows:
    result_df = pd.DataFrame(duplicated_rows)
else:
    result_df = pd.DataFrame(columns=df.columns)

# 6. 결과 확인 및 저장
result_df.to_csv('/home/psh/benchmark_after210930/meta_modif/metadata_duplicated.csv', index=False)

8hs2_B_C_R 관련 파일 개수: 3


## visualize (save pse file)

In [5]:
import glob
import os
from pymol import cmd

cmd.reinitialize()

## get point mutation 
root_dir = "/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v2.3.2.1_stage_2_wt_confidence_lr_1e-3/2025-10-26_10-47-26/epoch=42-step=61619/only_ab/1aqk_H_L_#"
align_num = 20

files = []
samples = []
count = 0

for sample in os.listdir(root_dir):
    if 'sample' not in sample:
        continue
    count += 1
    sample_path = os.path.join(root_dir, sample, "sample_1.pdb")
    samples.append(sample)
    files.append(sample_path)
    if count == align_num:
        break

# load sample files
for i, f in enumerate(files):
    cmd.load(f, f'sample_{i}')

#########################################################################################################
## Load WT
wt_path = "/home/psh/data/only_ab_valid/pdb/1aqk_H_L_#.pdb"
cmd.load(wt_path, 'wt')

#########################################################################################################
## Align all samples to WT

for i in range(len(files)):
    moving = f"sample_{i}"
    cmd.align(moving, "wt")

#########################################################################################################
## random coloring
cmd.util.cbc()

## save aligned session
cmd.save(f"{root_dir}/aligned_all.pse")
